In [4]:
import pandas as pd
import altair as alt

print("Pandas:", pd.__version__)
print("Altair:", alt.__version__)

Pandas: 2.1.4
Altair: 6.2.2


In [3]:
%pip install altair pandas ipykernel


   ---------------------------------------- 0.0/797.6 kB ? eta -:--:--
   -- ------------------------------------ 41.0/797.6 kB 960.0 kB/s eta 0:00:01
   ---------------- ----------------------- 337.9/797.6 kB 4.2 MB/s eta 0:00:01
   ---------------------------------------  788.5/797.6 kB 6.2 MB/s eta 0:00:01
   ---------------------------------------- 797.6/797.6 kB 5.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/501.7 kB ? eta -:--:--
   --------------------------------------- 501.7/501.7 kB 32.8 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [6]:
import pandas as pd
import altair as alt

url = "https://raw.githubusercontent.com/UIUC-iSchool-DataViz/is445_data/main/ufo-scrubbed-geocoded-time-standardized-00.csv"

columns = [
    "datetime", "city", "state", "country", "shape",
    "duration_seconds", "duration_hours_min", "comments",
    "date_posted", "latitude", "longitude"
]

df = pd.read_csv(
    url,
    header=None,
    names=columns,
    low_memory=False,
    on_bad_lines="skip",
    dtype=str
)

print(df.shape)
df.head()

(80332, 11)


,datetime,city,state,country,shape,duration_seconds,duration_hours_min,comments,date_posted,latitude,longitude
0,10/10/1949 20:30,san marcos,tx,us,cylinder,2700,45 minutes,This event took place in early fall around 194...,4/27/2004,29.8830556,-97.9411111
1,10/10/1949 21:00,lackland afb,tx,NaN,light,7200,1-2 hrs,1949 Lackland AFB&#44 TX. Lights racing acros...,12/16/2005,29.38421,-98.581082
2,10/10/1955 17:00,chester (uk/england),NaN,gb,circle,20,20 seconds,Green/Orange circular disc over Chester&#44 En...,1/21/2008,53.2,-2.916667
3,10/10/1956 21:00,edna,tx,us,circle,20,1/2 hour,My older brother and twin sister were leaving ...,1/17/2004,28.9783333,-96.6458333
4,10/10/1960 20:00,kaneohe,hi,us,light,900,15 minutes,AS a Marine 1st Lt. flying an FJ4B fighter/att...,1/22/2004,21.4180556,-157.8036111


In [7]:
# Extract year
df["year"] = pd.to_numeric(
    df["datetime"].str.extract(
        r"^\s*\d{1,2}/\d{1,2}/(\d{4})",
        expand=False
    ),
    errors="coerce"
)

# Clean shape names
df["shape"] = df["shape"].str.strip().str.lower()

# Remove missing values
df_clean = df.dropna(subset=["year", "shape"]).copy()
df_clean = df_clean[
    (df_clean["year"] >= 1900) &
    (df_clean["year"] <= 2026) &
    (df_clean["shape"] != "")
].copy()

df_clean["year"] = df_clean["year"].astype(int)

# Reports per year
yearly = (
    df_clean.groupby("year")
    .size()
    .reset_index(name="reports")
)

# Top 10 UFO shapes
top_shapes = df_clean["shape"].value_counts().head(10).index.tolist()

shape_year = (
    df_clean[df_clean["shape"].isin(top_shapes)]
    .groupby(["year", "shape"])
    .size()
    .reset_index(name="reports")
)

print("Total records:", len(df_clean))
print("Year range:", yearly["year"].min(), yearly["year"].max())
print("Top shapes:", top_shapes)

Total records: 78400
Year range: 1906 2014
Top shapes: ['light', 'triangle', 'circle', 'fireball', 'other', 'unknown', 'sphere', 'disk', 'oval', 'formation']


In [8]:
chart1 = (
    alt.Chart(yearly)
    .mark_line(
        color="#2563eb",
        strokeWidth=2.5,
        point=True
    )
    .encode(
        x=alt.X("year:O", title="Year"),
        y=alt.Y("reports:Q", title="Number of Reports"),
        tooltip=[
            alt.Tooltip("year:O", title="Year"),
            alt.Tooltip("reports:Q", title="Reports")
        ]
    )
    .properties(
        title="UFO Sightings Over Time",
        width=750,
        height=350
    )
)

chart1

alt.Chart(...)

In [9]:
years = sorted(shape_year["year"].unique().tolist())

dropdown = alt.binding_select(
    options=years,
    name="Select Year: "
)

selection = alt.selection_point(
    fields=["year"],
    bind=dropdown,
    value=years[-1]
)

chart2 = (
    alt.Chart(shape_year)
    .mark_bar()
    .encode(
        x=alt.X("reports:Q", title="Number of Reports"),
        y=alt.Y("shape:N", title="UFO Shape", sort="-x"),
        color=alt.Color(
            "shape:N",
            scale=alt.Scale(scheme="tableau10")
        ),
        tooltip=["year:O", "shape:N", "reports:Q"]
    )
    .add_params(selection)
    .transform_filter(selection)
    .properties(
        title="UFO Shapes by Selected Year",
        width=750,
        height=380
    )
)

chart2

alt.Chart(...)

In [10]:
from pathlib import Path

root = Path.cwd()

if root.name == "python_notebooks":
    root = root.parent

output_dir = root / "assets" / "hw5"
output_dir.mkdir(parents=True, exist_ok=True)

chart1.save(output_dir / "chart1.json")
chart2.save(output_dir / "chart2.json")

print("Saved to:", output_dir)

Saved to: d:\Projects\is455\aaa-jhb.github.io\assets\hw5
